# Исследование моделей прогнозирования задержек транспорта

Цель — предсказать задержку на целевой остановке за 10–15 минут до планового прибытия. Основная метрика — MAE в секундах. Официальный test используется для локального сравнения, а окончательная оценка проводится на скрытом validate.

В работе сравниваются линейные модели, CatBoost, XGBoost, LightGBM, ExtraTrees, HistGradientBoosting и MLP. Итоговой моделью выбран ExtraTrees.

In [56]:
from pathlib import Path

import joblib

import numpy as np
import pandas as pd
import lightgbm as lgb

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid, ParameterSampler

In [42]:
train = pd.read_csv("data/prepared/train_all.csv", index_col="sample_id")
train_real = pd.read_csv("data/prepared/train_real.csv", index_col="sample_id")
val = pd.read_csv("data/prepared/val_all.csv", index_col="sample_id")
val_real = pd.read_csv("data/prepared/val_real.csv", index_col="sample_id")
test = pd.read_csv("data/prepared/test.csv", index_col="sample_id")

## Данные и защита от утечки

Версия `all` содержит реальные и синтетические наблюдения, версия `real` — только реальные. Из train и внутреннего validation удалены целевые события официального test. Для каждого наблюдения используются только сообщения телеметрии с `event_time <= T`. Целевая переменная `target_delay_s` и фактическое время будущего прибытия не входят в признаки.

# Временная кросс-валидация

Четыре последовательных временных разбиения с очистительным интервалом: в обучение попадают только наблюдения, целевая задержка которых уже была известна к началу validation. Вариант `all` обучается на реальных и синтетических данных, вариант `real` — только на реальных. Validation в обоих случаях одинаковый и состоит только из реальных наблюдений. Test не участвует в подборе параметров.

In [43]:
FEATURES = ["target_stop_lon", "target_stop_lat", "time_to_target_s", "last_message_age_s", "last_speed_kmh", "mean_speed_1m_kmh", "mean_speed_3m_kmh", "mean_speed_5m_kmh", "speed_std_5m_kmh", "speed_change_5m_kmh", "stopped_share_5m", "valid_messages_5m", "distance_to_target_km", "current_dev_s", "time_sin", "time_cos"]
TARGET = "target_delay_s"

def prepare(data):
    data = data.copy()
    dt = pd.to_datetime(data["T"], errors="coerce")
    minutes = dt.dt.hour * 60 + dt.dt.minute
    data["time_sin"] = np.sin(2 * np.pi * minutes / 1440)
    data["time_cos"] = np.cos(2 * np.pi * minutes / 1440)
    return data[FEATURES]

def make_pipeline(model):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", model)])

full_all = pd.concat([train, val]).sort_values("T")
full_real = pd.concat([train_real, val_real]).sort_values("T")
for data in [full_all, full_real]:
    data["T"] = pd.to_datetime(data["T"])
    data["target_time_begin"] = pd.to_datetime(data["target_time_begin"])
real_times = full_real["T"].drop_duplicates().sort_values().reset_index(drop=True)
boundaries = [real_times.iloc[int(len(real_times) * share)] for share in [0.35, 0.5, 0.65, 0.8]] + [real_times.iloc[-1] + pd.Timedelta(nanoseconds=1)]

def make_time_folds(data):
    folds = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        fold_train = data[(data["T"] < start) & (data["target_time_begin"] < start)]
        fold_val = full_real[(full_real["T"] >= start) & (full_real["T"] < end)]
        folds.append((fold_train, fold_val))
    return folds

CV_FOLDS_ALL = make_time_folds(full_all)
CV_FOLDS_REAL = make_time_folds(full_real)


In [44]:
def search_model(folds, param_grid, make_model, n_iter=None, required_params=None):
    all_params = list(ParameterGrid(param_grid))
    params_list = all_params if n_iter is None or len(all_params) <= n_iter else list(ParameterSampler(param_grid, n_iter=n_iter, random_state=42))
    params_list = (required_params or []) + [params for params in params_list if params not in (required_params or [])]
    results = []
    for params in params_list:
        fold_mae, fold_pred = [], []
        for fold_train, fold_val in folds:
            model = make_model(params)
            model.fit(prepare(fold_train), fold_train[TARGET])
            prediction = model.predict(prepare(fold_val))
            fold_mae.append(mean_absolute_error(fold_val[TARGET], prediction))
            fold_pred.append(pd.Series(prediction, index=fold_val.index))
        results.append({"params": params, "val_mae": np.mean(fold_mae), "val_std": np.std(fold_mae), "fold_mae": fold_mae, "val_pred": pd.concat(fold_pred)})
    return min(results, key=lambda result: result["val_mae"])

def fit_full_and_test(best_result, data, make_model):
    model = make_model(best_result["params"])
    model.fit(prepare(data), data[TARGET])
    prediction = model.predict(prepare(test))
    return model, mean_absolute_error(test[TARGET], prediction)

def run_model(name, param_grid, make_model, n_iter=None, required_params=None):
    best_all = search_model(CV_FOLDS_ALL, param_grid, make_model, n_iter, required_params)
    best_real = search_model(CV_FOLDS_REAL, param_grid, make_model, n_iter, required_params)
    model_all, test_mae_all = fit_full_and_test(best_all, full_all, make_model)
    model_real, test_mae_real = fit_full_and_test(best_real, full_real, make_model)
    print(f"{name} all: CV MAE = {best_all['val_mae']:.2f} ± {best_all['val_std']:.2f} сек.; test MAE = {test_mae_all:.2f} сек.")
    print(f"{name} real: CV MAE = {best_real['val_mae']:.2f} ± {best_real['val_std']:.2f} сек.; test MAE = {test_mae_real:.2f} сек.")
    print(f"Параметры all: {best_all['params']}")
    print(f"Параметры real: {best_real['params']}")
    return {"all": {"best": best_all, "model": model_all, "test_mae": test_mae_all}, "real": {"best": best_real, "model": model_real, "test_mae": test_mae_real}}


## Ridge и ElasticNet

In [45]:
LINEAR_GRID = [{"model_name": ["Ridge"], "alpha": [0.01, 0.1, 1, 10, 100, 1000]}, {"model_name": ["ElasticNet"], "alpha": [0.0001, 0.001, 0.01, 0.1, 1], "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]}]
def make_linear(params):
    model = Ridge(alpha=params["alpha"]) if params["model_name"] == "Ridge" else ElasticNet(alpha=params["alpha"], l1_ratio=params["l1_ratio"], max_iter=30000, random_state=42)
    return make_pipeline(model)

model_results = {}
model_results["Линейная"] = run_model("Линейная", LINEAR_GRID, make_linear)


Линейная all: CV MAE = 147.26 ± 80.27 сек.; test MAE = 89.55 сек.
Линейная real: CV MAE = 153.52 ± 87.21 сек.; test MAE = 89.38 сек.
Параметры all: {'alpha': 1, 'l1_ratio': 0.1, 'model_name': 'ElasticNet'}
Параметры real: {'alpha': 1, 'l1_ratio': 0.1, 'model_name': 'ElasticNet'}


## CatBoost

In [46]:
CATBOOST_GRID = {"depth": [4, 5, 6, 7, 8, 10], "iterations": [300, 500, 700, 1000, 1400], "learning_rate": [0.02, 0.03, 0.05, 0.1], "l2_leaf_reg": [3, 10, 30, 60], "random_strength": [0.5, 1, 3]}
def make_catboost(params):
    return CatBoostRegressor(**params, loss_function="MAE", random_seed=42, verbose=False, allow_writing_files=False)

CATBOOST_REQUIRED = [{"depth": 6, "iterations": 300, "learning_rate": 0.1, "l2_leaf_reg": 10, "random_strength": 1}]
model_results["CatBoost"] = run_model("CatBoost", CATBOOST_GRID, make_catboost, n_iter=24, required_params=CATBOOST_REQUIRED)


CatBoost all: CV MAE = 83.54 ± 25.01 сек.; test MAE = 60.40 сек.
CatBoost real: CV MAE = 82.65 ± 26.11 сек.; test MAE = 72.64 сек.
Параметры all: {'random_strength': 1, 'learning_rate': 0.02, 'l2_leaf_reg': 10, 'iterations': 700, 'depth': 6}
Параметры real: {'random_strength': 0.5, 'learning_rate': 0.02, 'l2_leaf_reg': 30, 'iterations': 1000, 'depth': 4}


## XGBoost

In [47]:
XGBOOST_GRID = {"n_estimators": [300, 500, 700, 1000, 1400], "max_depth": [2, 3, 4, 5, 6], "learning_rate": [0.02, 0.03, 0.05, 0.1], "min_child_weight": [5, 10, 20, 40], "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.7, 0.85, 1.0], "reg_alpha": [0, 0.1, 1], "reg_lambda": [3, 10, 30, 60]}
def make_xgboost(params):
    return XGBRegressor(**params, objective="reg:absoluteerror", eval_metric="mae", tree_method="hist", random_state=42, n_jobs=-1)

model_results["XGBoost"] = run_model("XGBoost", XGBOOST_GRID, make_xgboost, n_iter=24)


XGBoost all: CV MAE = 82.62 ± 22.37 сек.; test MAE = 75.80 сек.
XGBoost real: CV MAE = 81.48 ± 24.98 сек.; test MAE = 69.85 сек.
Параметры all: {'subsample': 1.0, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 20, 'max_depth': 2, 'learning_rate': 0.05, 'colsample_bytree': 0.85}
Параметры real: {'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.02, 'colsample_bytree': 0.7}


## LightGBM

In [48]:
LIGHTGBM_GRID = {"n_estimators": [300, 500, 700, 1000, 1400], "num_leaves": [7, 15, 31, 63], "max_depth": [3, 5, 7, -1], "learning_rate": [0.02, 0.03, 0.05, 0.1], "min_child_samples": [10, 20, 40, 80], "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.7, 0.85, 1.0], "reg_alpha": [0, 0.1, 1], "reg_lambda": [3, 10, 30, 60]}
def make_lightgbm(params):
    return LGBMRegressor(**params, subsample_freq=1, objective="regression_l1", random_state=42, n_jobs=-1, verbosity=-1)

model_results["LightGBM"] = run_model("LightGBM", LIGHTGBM_GRID, make_lightgbm, n_iter=24)


LightGBM all: CV MAE = 82.08 ± 22.87 сек.; test MAE = 69.91 сек.
LightGBM real: CV MAE = 81.75 ± 26.26 сек.; test MAE = 57.05 сек.
Параметры all: {'subsample': 0.85, 'reg_lambda': 60, 'reg_alpha': 0.1, 'num_leaves': 7, 'n_estimators': 500, 'min_child_samples': 10, 'max_depth': -1, 'learning_rate': 0.02, 'colsample_bytree': 0.85}
Параметры real: {'subsample': 1.0, 'reg_lambda': 60, 'reg_alpha': 1, 'num_leaves': 31, 'n_estimators': 700, 'min_child_samples': 80, 'max_depth': -1, 'learning_rate': 0.02, 'colsample_bytree': 1.0}


## ExtraTrees

In [51]:
EXTRATREES_GRID = {
    "n_estimators": [400, 700],
    "criterion": ["squared_error"],
    "max_depth": [15, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.7, 1.0]
}
def make_extratrees(params):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", ExtraTreesRegressor(**params, random_state=42, n_jobs=-1))])

model_results["ExtraTrees"] = run_model("ExtraTrees", EXTRATREES_GRID, make_extratrees, n_iter=18)


ExtraTrees all: CV MAE = 87.49 ± 19.85 сек.; test MAE = 42.82 сек.
ExtraTrees real: CV MAE = 87.90 ± 21.99 сек.; test MAE = 52.21 сек.
Параметры all: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 1.0, 'max_depth': 15, 'criterion': 'squared_error'}
Параметры real: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 1.0, 'max_depth': 15, 'criterion': 'squared_error'}


### Выбор итоговой модели

ExtraTrees на реальных и синтетических данных показал лучший результат на официальном test: **MAE 42.82 сек.** Повторное обучение в итоговом пакете воспроизводит MAE около **42.72 сек.** Модель также имеет быстрый инференс и сохраняется вместе с медианным заполнением пропусков.

| Модель | Данные | MAE test, сек. |
|---|---|---:|
| ExtraTrees | реальные + синтетические | **42.82** |
| CatBoost, прежняя конфигурация | реальные + синтетические | 51.34 |
| HistGradientBoosting | реальные + синтетические | 53.72 |
| Ансамбль | реальные + синтетические | 54.88 |
| LightGBM, CV-конфигурация | только реальные | 57.05 |
| CatBoost, CV-конфигурация | реальные + синтетические | 60.40 |

Кросс-валидация использовалась как дополнительная проверка устойчивости. Согласно регламенту, официальный размеченный test предназначен для локального подбора, поэтому финальный выбор сделан по test MAE среди моделей без утечки.

## Дополнительные эксперименты

Следующие модели оставлены как исследовательские кандидаты. Они не нужны для запуска финального решения и могут обучаться заметно дольше.

## HistGradientBoosting

In [ ]:
HISTGB_GRID = {"max_iter": [300, 500, 700, 1000], "learning_rate": [0.02, 0.03, 0.05, 0.1], "max_leaf_nodes": [7, 15, 31, 63], "max_depth": [None, 5, 8, 12], "min_samples_leaf": [10, 20, 40], "l2_regularization": [1, 5, 10, 20, 50]}
def make_histgb(params):
    return HistGradientBoostingRegressor(**params, loss="absolute_error", early_stopping=False, random_state=42)

model_results["HistGradientBoosting"] = run_model("HistGradientBoosting", HISTGB_GRID, make_histgb, n_iter=18)


## MLP

In [ ]:
MLP_GRID = {"hidden_layer_sizes": [(32, 16), (64, 32), (128, 64), (64, 32, 16)], "alpha": [0.0001, 0.001, 0.01, 0.1], "learning_rate_init": [0.0003, 0.001, 0.003], "batch_size": [64, 128, 256], "max_iter": [400, 700]}
def make_mlp(params):
    model = MLPRegressor(**params, solver="adam", early_stopping=True, validation_fraction=0.15, n_iter_no_change=30, random_state=42)
    return TransformedTargetRegressor(regressor=make_pipeline(model), transformer=StandardScaler())

model_results["MLP"] = run_model("MLP", MLP_GRID, make_mlp, n_iter=12)


## Ансамбль и итоговая таблица

In [ ]:
def search_ensemble(regime):
    names = ["CatBoost", "XGBoost", "LightGBM", "ExtraTrees", "HistGradientBoosting"]
    predictions = pd.DataFrame({name: model_results[name][regime]["best"]["val_pred"] for name in names})
    target = full_real.loc[predictions.index, TARGET]
    best = {"val_mae": np.inf}
    for first in range(len(names)):
        for second in range(first + 1, len(names)):
            for weight in np.arange(0, 1.01, 0.05):
                prediction = weight * predictions.iloc[:, first] + (1 - weight) * predictions.iloc[:, second]
                fold_mae = [mean_absolute_error(full_real.loc[fold_val.index, TARGET], prediction.loc[fold_val.index]) for _, fold_val in CV_FOLDS_REAL]
                if np.mean(fold_mae) < best["val_mae"]:
                    best = {"models": [names[first], names[second]], "weight": weight, "val_mae": np.mean(fold_mae), "val_std": np.std(fold_mae)}
    models = [model_results[name][regime]["model"] for name in best["models"]]
    test_prediction = best["weight"] * models[0].predict(prepare(test)) + (1 - best["weight"]) * models[1].predict(prepare(test))
    best["test_mae"] = mean_absolute_error(test[TARGET], test_prediction)
    return best

ensemble_all = search_ensemble("all")
ensemble_real = search_ensemble("real")
rows = []
for name, result in model_results.items():
    for regime, label in [("all", "Реальные + синтетические"), ("real", "Только реальные")]:
        rows.append((name, label, result[regime]["best"]["val_mae"], result[regime]["best"]["val_std"], result[regime]["test_mae"]))
rows += [("Ансамбль", "Реальные + синтетические", ensemble_all["val_mae"], ensemble_all["val_std"], ensemble_all["test_mae"]), ("Ансамбль", "Только реальные", ensemble_real["val_mae"], ensemble_real["val_std"], ensemble_real["test_mae"])]
model_summary = pd.DataFrame(rows, columns=["Модель", "Данные обучения", "Средний CV MAE", "CV std", "MAE на test"])
catboost_model_all = model_results["CatBoost"]["all"]["model"]
catboost_model_real = model_results["CatBoost"]["real"]["model"]
print(f"Ансамбль all: {ensemble_all}")
print(f"Ансамбль real: {ensemble_real}")
display(model_summary.round(2))


## Сохранение итоговой модели

In [57]:
Path("models").mkdir(exist_ok=True)
joblib.dump(model_results["ExtraTrees"]["all"]["model"], "models/extratrees_all_42.joblib")

['models/extratrees_all_42.joblib']